In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.utils import AnalysisException
from delta.tables import DeltaTable
from utils import *
import sys,json,yaml,os
from pathlib import Path
import importlib.util

In [0]:
logger = get_logger("gold_notebook")
sys.path.append("/Workspace/Users/mayur10594@gmail.com/ETL_project")

In [0]:
# dbutils.widgets.text("fileList", "")
dbutils.widgets.text("taxYear", "")
dbutils.widgets.text("clientId", "")
dbutils.widgets.text("env","")
# dbutils.widgets.text("bronze_path", "")
dbutils.widgets.text("maps","")
# fileList = dbutils.widgets.get("fileList")
taxYear = dbutils.widgets.get("taxYear")
clientId = dbutils.widgets.get("clientId")
env = dbutils.widgets.get("env")
# bronze_path = dbutils.widgets.get("bronze_path")
maps=json.loads(dbutils.widgets.get("maps"))
logger.info(f"taxYear: {taxYear}, clientId: {clientId}, env: {env}, maps to be loaded: {maps}")

In [0]:
env_config=load_config(f"{root_dir}/config/etl_main.yaml",env)
sqls_config=load_config(f"{root_dir}/config/gold_config.yaml",env)

In [0]:
# Dictionary to hold DataFrames
dfs = {}
# create dataframe for all the maps
for map_name in maps:
    table=f"cp_database.{clientId}_{map_name}"
    df = spark.read.table(table)
    dfs[map_name] = df
    #print(f"df_{map_name} is created and stored in dfs['{map_name}']")
    logger.info(f"df_{map_name} is created and stored in dfs['{map_name}']")

In [0]:
extracts = []
df_ext = {}
for map in maps:
    dfs[map].createOrReplaceTempView(map)
    print(f"✅ Temp view created for {map}")
    logger.info(f"✅ Temp view created for {map}")

for map in maps:
    # Register input DF as temp view
    # dfs[map].createOrReplaceTempView(map)
    print(f"▶️ Started extract for {map}")
    sqls = sqls_config[map].get('sqls',[])
    transformations=sqls_config[map].get('transformations',[])
    trans_inputs=sqls_config[map].get('trans_inputs',[])
    input_dfs = [dfs[name] for name in trans_inputs]
    for sql_file in sqls:
        df_name = f"{map}_{sql_file.split('.')[0]}"   # string name
        extracts.append(df_name)
        print(f"▶️ Creating {df_name} dataframe")
        print(f"▶️ Running SQL for {map}: {sql_file}")
        # Load the SQL text
        sql_query = load_sqls(f"{root_dir}gold_transformations", sql_file)
        print(sql_query)
        # Run SQL and store DF in dictionary
        df_ext[df_name] = spark.sql(sql_query)
        df_ext[df_name].write.format("parquet").mode('overwrite').save(f"{env_config['gold_path']}extracts/{map}/{sql_file}")
    for trans_file in transformations:
        print(f"▶️ Running transformation for {map}: {trans_file}")
        df_name = f"{map}_{trans_file.split('.')[0]}"   # string name
        extracts.append(df_name)
        print(f"▶️ Creating {df_name} dataframe")
        print(f"▶️ Running transformation for {map}: {trans_file}")
        trans_folder = Path("../gold_transformations/")
        # Transformation filename from YAML
        script_path = Path(trans_folder / trans_file)
        spec = importlib.util.spec_from_file_location(script_path.stem, script_path)
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        # Get function object (assume function name = filename without .py)
        func = getattr(module, script_path.stem)
        df_ext[df_name]=func(*input_dfs)
        df_ext[df_name].write.format("parquet").mode('overwrite').save(f"{env_config['gold_path']}extracts/{map}/{trans_file}")
 

print("✅ Extracts created:", extracts)


In [0]:
# df_ext['users_users_extract']=df_ext['users_users_extract'].union(df_new)

In [0]:
# df_ext['users_users_extract']=df_ext['users_users_extract'].withColumn('UserName', when(col('UserId')=='100',lit('Dhanashri Sonawane')).otherwise(col('UserName')))

In [0]:
# df_new=df_new.withColumn('UserId',col('UserId')+'10000').withColumn('UserName', lit('Test_UserName'))

In [0]:
for name,df in df_ext.items():
    if (name.split("_")[0]=='cards') & (name.split("_")[1]=='debit'):
        print("this is for CardsDebit table")
        df_ext[name].write.mode("overwrite").saveAsTable(f"gold_database.{clientId}_CardsDebit")
    elif (name.split("_")[0]=='cards') & (name.split("_")[1]=='credit'):
        print("this is for CardsCredit table")
        df_ext[name].write.mode("overwrite").saveAsTable(f"gold_database.{clientId}_CardsCredit")
    elif (name.split("_")[0]=='transactions') & (name.split("_")[1]=='debit'):
        print("this is for TransDataDebit table")
        df_ext[name].write.mode("append").saveAsTable(f"gold_database.{clientId}_TransDataDebit")
    elif (name.split("_")[0]=='transactions') & (name.split("_")[1]=='credit'):
        print("this is for TransDataCredit table")
        df_ext[name].write.mode("append").saveAsTable(f"gold_database.{clientId}_TransDataCredit")
    elif (name.split("_")[0]=='transactions') & (name.split("_")[1]=='transactions'):
        print("this is for TransData table")
        df_ext[name].write.mode("append").saveAsTable(f"gold_database.{clientId}_TransData")
    elif name.split("_")[0]=='users':
        print("this is for Users table")
        try:
            DeltaTable.forPath(spark, f"gold_database.{clientId}_users")
            print("Table already exists, proceeding with SCD2")
            run_scd_type2(
                spark,
                df_source=df_ext[name],
                target_table_path=f"gold_database.{clientId}_users",
                config_path="/Workspace/Users/mayur10594@gmail.com/ETL_project/config/scd_config.yaml"
                )
        # df_ext[name].write.mode("overwrite").saveAsTable(f"gold_database.{clientId}_Users")
        except AnalysisException:
            print(f"⚠️ Delta table not found at {f"gold_database.{clientId}_users"} — creating it now.")
            df_ext[name].write.mode("overwrite").saveAsTable(f"gold_database.{clientId}_Users")
    else:
        print("this is for something else, can not proceed for this dataframe")
print("All dataframes are written to gold_database")

In [0]:
# targetDelta = DeltaTable.forName(spark, f"gold_database.{clientId}_users")

In [0]:
# df_target = targetDelta.toDF()
# df_target.filter(col('ActiveStatus') != "D").count()

In [0]:
# df_source=df_ext['users_users_extract']

In [0]:
%sql use gold_database;
select * from se2_users where UserId in ('100','1000');
-- select count(*) from se2_transactions;
-- select count(*) from se2_users;

In [0]:
logger.info(f"exiting the gold_notetbook notebook")
dbutils.notebook.exit(json.dumps({
    "status": "OK",
    "message": "All files processed successfully"
}))